In [ ]:
# ============================================================
# COMPLETE PIPELINE - UPLOAD 3 DATASETS + GET MODEL METRICS
# ============================================================

# Step 1: Install required packages
!pip install ultralytics pandas --quiet

# Step 2: Import libraries
from google.colab import files
import zipfile
import os
import glob
import yaml
import shutil
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Step 3: Upload THREE dataset zip files
# ------------------------------------------------------------
print("="*60)
print("📤 UPLOAD YOUR THREE DATASET ZIP FILES")
print("="*60)
print("Please upload all 3 datasets:")
print(" - V21 dataset zip")
print(" - V24 dataset zip")
print(" - V25 dataset zip")
print("(You can select multiple files at once)")

uploaded = files.upload()

# ------------------------------------------------------------
# Step 4: Create directories for datasets
# ------------------------------------------------------------
!mkdir -p /content/datasets/v21
!mkdir -p /content/datasets/v24
!mkdir -p /content/datasets/v25
!mkdir -p /content/datasets/val/images  # For path fixing

# ------------------------------------------------------------
# Step 5: Extract and organize datasets
# ------------------------------------------------------------
dataset_paths = {}

for zip_filename in uploaded.keys():
    print(f"\n📁 Processing: {zip_filename}")

    # Determine which version this is
    if 'v21' in zip_filename.lower():
        version = 'v21'
    elif 'v24' in zip_filename.lower():
        version = 'v24'
    elif 'v25' in zip_filename.lower():
        version = 'v25'
    else:
        print(f"⚠️ Unknown version: {zip_filename}, skipping...")
        continue

    # Extract to version-specific folder
    extract_path = f"/content/datasets/{version}"
    !mkdir -p {extract_path}

    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    # Find data.yaml
    yaml_files = glob.glob(f"{extract_path}/**/data.yaml", recursive=True)

    if yaml_files:
        data_yaml_path = yaml_files[0]
        print(f"✅ {version} dataset ready")

        # Fix the data.yaml paths
        with open(data_yaml_path, 'r') as f:
            data = yaml.safe_load(f)

        data['path'] = '/content/datasets'
        data['train'] = f'{version}/train/images'
        data['val'] = f'{version}/valid/images'

        fixed_path = data_yaml_path.replace('data.yaml', f'data_fixed_{version}.yaml')
        with open(fixed_path, 'w') as f:
            yaml.dump(data, f, default_flow_style=False)

        dataset_paths[version] = fixed_path
        print(f"📄 {version} data.yaml contents:")
        !cat "{fixed_path}"
    else:
        print(f"❌ data.yaml not found in {zip_filename}!")

print("\n✅ DATASETS READY!")
print(dataset_paths)

# ------------------------------------------------------------
# Step 6: Mount Google Drive for models (once per session)
# ------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# Step 7: Import ML libraries and set model paths
# ------------------------------------------------------------
from ultralytics import YOLO

# Your model paths in Drive
model_paths = {
    'V21': "/content/drive/MyDrive/TeamProject/models/srilankan_food_model_v21_74.5.pt",
    'V24': "/content/drive/MyDrive/TeamProject/models/srilankan_food_model_v24_71.9.pt",
    'V25': "/content/drive/MyDrive/TeamProject/models/srilankan_food_model_v25_70.5.pt"
}

# Check if models exist
for name, path in model_paths.items():
    if os.path.exists(path):
        print(f"✅ {name} model found")
    else:
        print(f"❌ {name} model not found at: {path}")

# ------------------------------------------------------------
# Step 8: Evaluate models and get metrics
# ------------------------------------------------------------
results = {}

def get_model_metrics(model_path, model_name, data_yaml_path):
    print(f"\n🔍 Evaluating {model_name}...")
    model = YOLO(model_path)
    res = model.val(
        data=data_yaml_path,
        plots=True,
        save_json=True,
        conf=0.25,
        iou=0.5
    )
    print(f"✅ {model_name} - mAP50: {res.box.map50:.4f}")
    return res

for name in ['V21', 'V24', 'V25']:
    if name.lower() in dataset_paths:
        results[name] = get_model_metrics(model_paths[name], name, dataset_paths[name.lower()])

# ------------------------------------------------------------
# Step 9: Top 10 and Bottom 10 performing classes
# ------------------------------------------------------------
for name, res in results.items():
    # Handle unequal array lengths safely
    min_len = min(len(res.box.p), len(res.box.r), len(res.box.ap50), len(res.box.ap), len(res.names))

    df = pd.DataFrame({
        'Class': list(res.names.values())[:min_len],
        'Precision': res.box.p[:min_len],
        'Recall': res.box.r[:min_len],
        'mAP50': res.box.ap50[:min_len],
        'mAP50-95': res.box.ap[:min_len]
    }).sort_values('mAP50', ascending=False).reset_index(drop=True)

    print("\n" + "="*60)
    print(f"📊 {name} - TOP 10 CLASSES")
    print("="*60)
    print(df.head(10).to_string(index=False))

    print("\n" + "-"*60)
    print(f"📉 {name} - BOTTOM 10 CLASSES")
    print("-"*60)
    print(df.tail(10).to_string(index=False))

    # Save to CSV
    df.to_csv(f"{name}_performance.csv", index=False)

# ------------------------------------------------------------
# Step 10: Final comparison table (all metrics)
# ------------------------------------------------------------
comparison = pd.DataFrame({
    'Metric': ['mAP50', 'mAP50-95', 'Avg Precision', 'Avg Recall'],
    'V21': [results['V21'].box.map50, results['V21'].box.map, np.mean(results['V21'].box.p), np.mean(results['V21'].box.r)],
    'V24': [results['V24'].box.map50, results['V24'].box.map, np.mean(results['V24'].box.p), np.mean(results['V24'].box.r)],
    'V25': [results['V25'].box.map50, results['V25'].box.map, np.mean(results['V25'].box.p), np.mean(results['V25'].box.r)]
})

print("\n" + "="*60)
print("📊 FINAL MODEL COMPARISON")
print("="*60)
print(comparison.to_string(index=False))

# Save comparison CSV
comparison.to_csv("model_comparison.csv", index=False)

print("\n✅ ALL RESULTS GENERATED SUCCESSFULLY!")
print("Saved files:")
print(" - V21_performance.csv")
print(" - V24_performance.csv")
print(" - V25_performance.csv")
print(" - model_comparison.csv")

# Optional: Download files
print("\n📥 Download results?")
download = input("Download files? (y/n): ").lower()
if download == 'y':
    files.download('V21_performance.csv')
    files.download('V24_performance.csv')
    files.download('V25_performance.csv')
    files.download('model_comparison.csv')

📤 UPLOAD YOUR THREE DATASET ZIP FILES
Please upload all 3 datasets:
 - V21 dataset zip
 - V24 dataset zip
 - V25 dataset zip
(You can select multiple files at once)


Saving food detect.v21-more_images_added_new2.yolov8.zip to food detect.v21-more_images_added_new2.yolov8 (2).zip
Saving food detect.v24-without_autoajustcontrast.yolov8.zip to food detect.v24-without_autoajustcontrast.yolov8 (2).zip
Saving food detect.v25-categories_removed_version.yolov8.zip to food detect.v25-categories_removed_version.yolov8 (2).zip

📁 Processing: food detect.v21-more_images_added_new2.yolov8 (2).zip
✅ v21 dataset ready
📄 v21 data.yaml contents:
names:
- Baked Filled
- Baked Sweet bun
- Beans curry
- Beetroot Curry
- Cabbage Curry
- Carrot
- Cashew Curry
- Chicken curry
- Cocount roti
- Dhal Curry
- Egg curry
- Fish Curry
- Fried Filled
- Fried rice
- Gotukola Mallum
- Hoppers
- Kiribath
- Kottu
- Lunu sambol
- Mango curry
- Moringa curry
- Okra curry
- Papadam
- Pittu
- Pol Sambol
- Polos curry
- Potato Curry
- Prawns Curry
- Red rice
- Sausage Hotdog
- Soya curry
- Sprats
- String Hoppers
- Sweets
- Wade
- Watalappam
- rice
- yellow rice
nc: 38
path: /content/dat

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>